# CASE 1 BDC — Fine-tuning Multi-Backbone untuk Klasifikasi Tweet MBG

Klasifikasi multikelas tweet berbahasa Indonesia mengenai program Makan Bergizi Gratis (MBG) ke dalam 8 kelas. Metrik evaluasi: **balanced accuracy**.

## Struktur Notebook

1. **Setup** (cell 1–4): install dependencies, load data, label encoding, definisi class Dataset, dan fungsi training.
2. **Cell training per model** (cell 7–12): satu cell per backbone — masing-masing menjalankan 5-fold cross-validation, mencetak laporan evaluasi lengkap (balanced accuracy, accuracy, macro/weighted F1, MAE/MSE pada label ordinal, precision/recall/F1 per kelas, confusion matrix), dan menyimpan prediksi OOF + test ke dalam registry.
    - Cell 7: `indolem/indobertweet-base-uncased` ← **Model domain Twitter; backbone tunggal terkuat untuk task ini**
    - Cell 8: `indobenchmark/indobert-base-p1` (opsional)
    - Cell 9: `indobenchmark/indobert-large-p1` (opsional)
    - Cell 10: `xlm-roberta-base` (opsional)
    - Cell 11: `xlm-roberta-large` (opsional)
    - Cell 12: `flax-community/indonesian-roberta-base` ← **Dipakai dalam submission final untuk ensemble bersama IndoBERTweet**
    - Cell 13 (opsional): **Stacking** — ekstrak embedding [CLS] dari transformer terbaik dan feed ke XGBoost. Menambah diversitas ensemble (dalam eksperimen kami, stacking justru menurunkan skor sehingga tidak dipakai di submission final).
3. **Perbandingan & seleksi** (cell 14–16): tabel ringkasan yang membandingkan semua model + ensemble, lalu **cell final** untuk memilih model yang dipakai dan menghasilkan file submission.

**Tips:** Tidak perlu menjalankan semua backbone. Skip cell yang waktu eksekusinya tidak memungkinkan — cell final hanya menggunakan model yang ada di registry. Pada T4 GPU (Colab gratis), urutan yang direkomendasikan: jalankan **Cell 7 (IndoBERTweet) dan Cell 12 (IndoRoBERTa) terlebih dahulu**, lalu Cell 16 untuk menghasilkan submission. Total waktu ~25 menit.

**Mengapa IndoBERTweet dipilih?** Model ini di-pretrain pada teks Twitter berbahasa Indonesia — slang, singkatan, gaya bahasa informal yang persis dengan karakteristik dataset MBG. Pada eksperimen sebelumnya, skor balanced accuracy mencapai ~0.70 hanya dengan single split 80/20. Pada 5-fold CV, skor OOF stabil di 0.6893.

**Mengapa IndoRoBERTa ditambahkan ke ensemble?** IndoRoBERTa menggunakan objective pretraining yang berbeda (RoBERTa dynamic masking, tanpa NSP) dan tokenizer yang berbeda (byte-level BPE) dibanding IndoBERTweet (BERT MLM + NSP, WordPiece). Karena keduanya "melihat" data dengan cara yang berbeda, error pattern keduanya tidak terkorelasi — inilah syarat utama agar soft-vote ensemble efektif. Hasilnya terbukti: ensemble IndoBERTweet + IndoRoBERTa mencapai OOF 0.6914, mengungguli kedua single model.

**Kepatuhan aturan:** seluruh backbone yang digunakan adalah *encoder-only pretrained language model* yang di-*fine-tune* pada data panitia. Sesuai dengan aturan (e) pada Petunjuk Teknis, penggunaan model seperti BERT, IndoBERT, RoBERTa, dan XLM-R untuk fine-tuning diperbolehkan secara eksplisit. Tidak ada generative AI/LLM, tidak ada prediksi zero-shot atau few-shot, tidak ada data eksternal, tidak ada pseudo-labeling atau augmentasi label.

In [1]:
# CELL 1 — Install dependencies
!pip install -q transformers==4.44.2 accelerate==0.34.2 scikit-learn pandas openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 41.2 MB/s eta 0:00:00


In [2]:
# CELL 2 — Imports, seed, device
import os, re, html, random, gc, json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score, f1_score,
    precision_score, recall_score, classification_report,
    confusion_matrix, mean_absolute_error, mean_squared_error,
    cohen_kappa_score, matthews_corrcoef,
)
from google.colab import drive
drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'GPU mem: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Mounted at /content/drive
Device: cuda
GPU: Tesla T4
GPU mem: 15.6 GB


In [3]:
# CELL 3 — Load & clean data, label encoding
LABELED_PATH = '/content/drive/MyDrive/satria/case_1_labeled_data.xlsx'
PREDICT_PATH = '/content/drive/MyDrive/satria/case_1_text_to_predict.xlsx'
TEMPLATE_PATH = '/content/drive/MyDrive/satria/case_1_template_sheet.xlsx'

URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
MULTISPACE_RE = re.compile(r'\s+')

def clean(t):
    if not isinstance(t, str):
        return ''
    t = html.unescape(t).replace('&amp;', '&')
    t = URL_RE.sub(' [URL] ', t)
    t = MENTION_RE.sub(' [USER] ', t)
    t = MULTISPACE_RE.sub(' ', t).strip()
    return t

labeled = pd.read_excel(LABELED_PATH).drop_duplicates(subset=['full_text']).reset_index(drop=True)
predict_df = pd.read_excel(PREDICT_PATH)
template = pd.read_excel(TEMPLATE_PATH)
labeled['text'] = labeled['full_text'].map(clean)
predict_df['text'] = predict_df['full_text'].map(clean)

LABELS = sorted(labeled['label'].unique())
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
labeled['y'] = labeled['label'].map(label2id)
N_CLASSES = len(LABELS)

print(f'Train: {len(labeled)}   Predict: {len(predict_df)}   Classes: {N_CLASSES}')
print('\nClass distribution:')
print(labeled['label'].value_counts())
print('\nLabel encoding:'); print(label2id)

# Class weights (inverse-frequency) — used in weighted CE loss to combat
# imbalance, since balanced_accuracy is the eval metric.
counts = labeled['y'].value_counts().sort_index().values
weights_np = len(labeled) / (N_CLASSES * counts)
class_weights = torch.tensor(weights_np, dtype=torch.float32).to(device)
print('\nClass weights:'); print(dict(zip(LABELS, weights_np.round(3))))

Train: 4999   Predict: 1500   Classes: 8

Class distribution:
label
Kualitas Pangan     1246
Politik              792
Anggaran             727
Lainnya              638
Tata Kelola          511
Sasaran Penerima     507
Distribusi           433
Ekonomi              145
Name: count, dtype: int64

Label encoding:
{'Anggaran': 0, 'Distribusi': 1, 'Ekonomi': 2, 'Kualitas Pangan': 3, 'Lainnya': 4, 'Politik': 5, 'Sasaran Penerima': 6, 'Tata Kelola': 7}

Class weights:
{'Anggaran': np.float64(0.86), 'Distribusi': np.float64(1.443), 'Ekonomi': np.float64(4.309), 'Kualitas Pangan': np.float64(0.502), 'Lainnya': np.float64(0.979), 'Politik': np.float64(0.789), 'Sasaran Penerima': np.float64(1.232), 'Tata Kelola': np.float64(1.223)}


In [4]:
# CELL 4 — Dataset class & training/inference functions
N_FOLDS = 5

class TweetDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len, labels=None):
        self.texts = list(texts)
        self.labels = list(labels) if labels is not None else None
        self.tok = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx], truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def train_one_fold(model_name, tokenizer, train_idx, val_idx, fold_id, *,
                   max_len, batch_size, lr, epochs, warmup_ratio,
                   weight_decay, grad_accum):
    """Train one fold. Returns (best_model, best_val_balanced_acc).
    Keeps the checkpoint with the highest validation balanced accuracy."""
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=N_CLASSES,
        id2label=id2label, label2id=label2id,
    ).to(device)

    tr_ds = TweetDataset(labeled.loc[train_idx, 'text'].values, tokenizer,
                         max_len, labeled.loc[train_idx, 'y'].values)
    va_ds = TweetDataset(labeled.loc[val_idx, 'text'].values, tokenizer,
                         max_len, labeled.loc[val_idx, 'y'].values)
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                       num_workers=2, pin_memory=True)
    va_ld = DataLoader(va_ds, batch_size=batch_size*2, shuffle=False,
                       num_workers=2, pin_memory=True)

    no_decay = ['bias', 'LayerNorm.weight']
    params = [
        {'params': [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay)],
         'weight_decay': weight_decay},
        {'params': [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    optim = AdamW(params, lr=lr)
    total_steps = len(tr_ld) * epochs // grad_accum
    sched = get_linear_schedule_with_warmup(
        optim, num_warmup_steps=int(total_steps * warmup_ratio),
        num_training_steps=total_steps,
    )
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    scaler = torch.cuda.amp.GradScaler()

    best_score = -1.0
    best_state = None
    best_epoch = -1
    for epoch in range(epochs):
        model.train()
        tot = 0.0
        optim.zero_grad()
        for step, batch in enumerate(tr_ld):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            with torch.cuda.amp.autocast(dtype=torch.float16):
                out = model(**{k: v for k, v in batch.items() if k != 'labels'})
                loss = loss_fn(out.logits, batch['labels']) / grad_accum
            scaler.scale(loss).backward()
            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optim); scaler.update()
                sched.step(); optim.zero_grad()
            tot += loss.item() * grad_accum

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in va_ld:
                batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                with torch.cuda.amp.autocast(dtype=torch.float16):
                    out = model(**{k: v for k, v in batch.items() if k != 'labels'})
                preds.append(out.logits.argmax(-1).cpu().numpy())
                trues.append(batch['labels'].cpu().numpy())
        preds = np.concatenate(preds); trues = np.concatenate(trues)
        score = balanced_accuracy_score(trues, preds)
        print(f'  [fold {fold_id}] epoch {epoch+1}/{epochs}  '
              f'train_loss={tot/len(tr_ld):.4f}  val_bal_acc={score:.4f}')
        if score > best_score:
            best_score = score
            best_epoch = epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f'  [fold {fold_id}] best epoch = {best_epoch}, best bal_acc = {best_score:.4f}')
    model.load_state_dict(best_state)
    return model, best_score


def predict_probs(model, tokenizer, texts, max_len, batch_size):
    ds = TweetDataset(texts, tokenizer, max_len)
    ld = DataLoader(ds, batch_size=batch_size*2, shuffle=False,
                    num_workers=2, pin_memory=True)
    model.eval()
    out_chunks = []
    with torch.no_grad():
        for batch in ld:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            with torch.cuda.amp.autocast(dtype=torch.float16):
                logits = model(**batch).logits
            out_chunks.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(out_chunks, axis=0)


def run_backbone(model_name, *, max_len=160, batch_size=16, lr=2e-5,
                 epochs=4, warmup_ratio=0.1, weight_decay=0.01, grad_accum=1):
    """Run K-fold training for one backbone. Returns dict with OOF + test
    probability matrices and per-fold scores."""
    print(f'\n{"="*70}\nBACKBONE: {model_name}\n{"="*70}')
    print(f'  max_len={max_len}  batch={batch_size}  lr={lr}  epochs={epochs}')

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros((len(labeled), N_CLASSES), dtype=np.float32)
    test_p = np.zeros((len(predict_df), N_CLASSES), dtype=np.float32)
    fold_scores = []

    test_texts = predict_df['text'].values
    for fold, (tr_idx, va_idx) in enumerate(skf.split(labeled['text'], labeled['y']), 1):
        print(f'\n--- Fold {fold}/{N_FOLDS} ---')
        model, score = train_one_fold(
            model_name, tokenizer, tr_idx, va_idx, fold,
            max_len=max_len, batch_size=batch_size, lr=lr, epochs=epochs,
            warmup_ratio=warmup_ratio, weight_decay=weight_decay,
            grad_accum=grad_accum,
        )
        fold_scores.append(score)
        oof[va_idx] = predict_probs(
            model, tokenizer, labeled.loc[va_idx, 'text'].values,
            max_len, batch_size,
        )
        test_p += predict_probs(model, tokenizer, test_texts, max_len, batch_size) / N_FOLDS
        del model; gc.collect(); torch.cuda.empty_cache()

    return {
        'name': model_name,
        'oof_probs': oof,
        'test_probs': test_p,
        'fold_scores': fold_scores,
        'config': dict(max_len=max_len, batch_size=batch_size, lr=lr,
                       epochs=epochs, warmup_ratio=warmup_ratio,
                       weight_decay=weight_decay, grad_accum=grad_accum),
    }


def evaluate(name, oof_probs, fold_scores=None):
    """Print a thorough evaluation report and return a metrics dict."""
    y_true = labeled['y'].values
    y_pred = oof_probs.argmax(1)

    metrics = {
        'name': name,
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted'),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'cohen_kappa': cohen_kappa_score(y_true, y_pred),
        'mcc': matthews_corrcoef(y_true, y_pred),
        # MAE / MSE on the label ordinals — informational only (labels aren't
        # truly ordinal, but the panitia mentioned MAE/MSE so we include them).
        'mae_on_ids': mean_absolute_error(y_true, y_pred),
        'mse_on_ids': mean_squared_error(y_true, y_pred),
        'fold_scores': fold_scores or [],
    }

    print(f'\n{"="*70}\nEVALUATION: {name}\n{"="*70}')
    if fold_scores:
        print(f'Per-fold balanced_acc : {[round(s,4) for s in fold_scores]}')
        print(f'Mean fold balanced_acc: {np.mean(fold_scores):.4f}  '
              f'(std {np.std(fold_scores):.4f})')
    print(f'\n--- OOF (out-of-fold) metrics ---')
    print(f'  Balanced accuracy : {metrics["balanced_accuracy"]:.4f}  <-- COMPETITION METRIC')
    print(f'  Plain accuracy    : {metrics["accuracy"]:.4f}')
    print(f'  Macro F1          : {metrics["macro_f1"]:.4f}')
    print(f'  Weighted F1       : {metrics["weighted_f1"]:.4f}')
    print(f'  Macro precision   : {metrics["macro_precision"]:.4f}')
    print(f'  Macro recall      : {metrics["macro_recall"]:.4f}')
    print(f'  Cohen kappa       : {metrics["cohen_kappa"]:.4f}')
    print(f'  Matthews corrcoef : {metrics["mcc"]:.4f}')
    print(f'  MAE (on label ids): {metrics["mae_on_ids"]:.4f}   (informational)')
    print(f'  MSE (on label ids): {metrics["mse_on_ids"]:.4f}   (informational)')

    print('\n--- Per-class report ---')
    print(classification_report(y_true, y_pred, target_names=LABELS, digits=4, zero_division=0))

    print('--- Confusion matrix (rows=true, cols=pred) ---')
    cm = confusion_matrix(y_true, y_pred)
    header = '             ' + ' '.join(f'{l[:8]:>8}' for l in LABELS)
    print(header)
    for i, row in enumerate(cm):
        print(f'{LABELS[i][:12]:<12} ' + ' '.join(f'{v:>8d}' for v in row))
    return metrics

# Global registry — every backbone cell appends here.
RESULTS = {}
print('Setup complete. Ready to train backbones.')

Setup complete. Ready to train backbones.


---
## Backbone training cells

Run each cell to train and evaluate that backbone. You can run them in any order, skip any you can't afford to train, or re-run with different hyperparameters. Each cell appends to `RESULTS`.


In [5]:
# CELL 7 — Backbone A: IndoBERTweet (Twitter-domain — typically strongest)
# max_len=128 matches the typical tweet length distribution
MODEL_TWEET = 'indolem/indobertweet-base-uncased'
RESULTS['indobertweet'] = run_backbone(
    MODEL_TWEET,
    max_len=128, batch_size=16, lr=2e-5, epochs=5,
    warmup_ratio=0.1, weight_decay=0.05, grad_accum=1,
)
RESULTS['indobertweet']['metrics'] = evaluate(
    MODEL_TWEET,
    RESULTS['indobertweet']['oof_probs'],
    RESULTS['indobertweet']['fold_scores'],
)


BACKBONE: indolem/indobertweet-base-uncased
  max_len=128  batch=16  lr=2e-05  epochs=5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

  [fold 5] epoch 4/5  train_loss=0.4957  val_bal_acc=0.6660


/tmp/ipykernel_2500/1216671756.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
/tmp/ipykernel_2500/1216671756.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


  [fold 5] epoch 5/5  train_loss=0.3887  val_bal_acc=0.6611
  [fold 5] best epoch = 4, best bal_acc = 0.6660


/tmp/ipykernel_2500/1216671756.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):



EVALUATION: indolem/indobertweet-base-uncased
Per-fold balanced_acc : [np.float64(0.6905), np.float64(0.6848), np.float64(0.7009), np.float64(0.7044), np.float64(0.666)]
Mean fold balanced_acc: 0.6893  (std 0.0136)

--- OOF (out-of-fold) metrics ---
  Balanced accuracy : 0.6893  <-- COMPETITION METRIC
  Plain accuracy    : 0.6749
  Macro F1          : 0.6759
  Weighted F1       : 0.6752
  Macro precision   : 0.6655
  Macro recall      : 0.6893
  Cohen kappa       : 0.6180
  Matthews corrcoef : 0.6185
  MAE (on label ids): 0.9508   (informational)
  MSE (on label ids): 3.7862   (informational)

--- Per-class report ---
                  precision    recall  f1-score   support

        Anggaran     0.7585    0.7992    0.7783       727
      Distribusi     0.6555    0.7252    0.6886       433
         Ekonomi     0.7202    0.8345    0.7732       145
 Kualitas Pangan     0.7706    0.6846    0.7250      1246
         Lainnya     0.6057    0.6019    0.6038       638
         Politik     0.6

In [6]:
# CELL 8 — Backbone B: IndoBERT base (general-domain)
MODEL_A = 'flax-community/indonesian-roberta-base'
RESULTS['indonesian-roberta-base'] = run_backbone(
    MODEL_A,
    max_len=160, batch_size=16, lr=2e-5, epochs=4,
    warmup_ratio=0.1, weight_decay=0.01, grad_accum=1,
)
RESULTS['indonesian-roberta-base']['metrics'] = evaluate(
    MODEL_A,
    RESULTS['indonesian-roberta-base']['oof_probs'],
    RESULTS['indonesian-roberta-base']['fold_scores'],
)


BACKBONE: flax-community/indonesian-roberta-base
  max_len=160  batch=16  lr=2e-05  epochs=4


tokenizer_config.json:   0%|          | 0.00/292 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  [fold 5] epoch 3/4  train_loss=0.7934  val_bal_acc=0.6340


/tmp/ipykernel_2500/1216671756.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
/tmp/ipykernel_2500/1216671756.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


  [fold 5] epoch 4/4  train_loss=0.6093  val_bal_acc=0.6435
  [fold 5] best epoch = 4, best bal_acc = 0.6435


/tmp/ipykernel_2500/1216671756.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):



EVALUATION: flax-community/indonesian-roberta-base
Per-fold balanced_acc : [np.float64(0.6429), np.float64(0.6632), np.float64(0.6892), np.float64(0.6575), np.float64(0.6435)]
Mean fold balanced_acc: 0.6593  (std 0.0169)

--- OOF (out-of-fold) metrics ---
  Balanced accuracy : 0.6593  <-- COMPETITION METRIC
  Plain accuracy    : 0.6461
  Macro F1          : 0.6459
  Weighted F1       : 0.6468
  Macro precision   : 0.6367
  Macro recall      : 0.6593
  Cohen kappa       : 0.5842
  Matthews corrcoef : 0.5847
  MAE (on label ids): 1.0310   (informational)
  MSE (on label ids): 4.0448   (informational)

--- Per-class report ---
                  precision    recall  f1-score   support

        Anggaran     0.7674    0.7579    0.7626       727
      Distribusi     0.6102    0.6905    0.6479       433
         Ekonomi     0.6706    0.7862    0.7238       145
 Kualitas Pangan     0.7266    0.6653    0.6946      1246
         Lainnya     0.5836    0.5470    0.5647       638
         Politik  

In [7]:
# CELL 9 — Backbone C: IndoBERT LARGE
# Notes:
#   - Lower LR (1e-5) for stability with the large model.
#   - Bumped epochs to 5 — large models sometimes need an extra epoch.
#   - If you hit OOM on T4, set batch_size=8 and grad_accum=2 (effective bs=16).
MODEL_B = 'cahya/roberta-base-indonesian-1.5G'
RESULTS['roberta-base-indonesian-1.5G'] = run_backbone(
    MODEL_B,
    max_len=160, batch_size=8, lr=1e-5, epochs=5,
    warmup_ratio=0.1, weight_decay=0.01, grad_accum=2,
)
RESULTS['roberta-base-indonesian-1.5G']['metrics'] = evaluate(
    MODEL_B,
    RESULTS['roberta-base-indonesian-1.5G']['oof_probs'],
    RESULTS['roberta-base-indonesian-1.5G']['fold_scores'],
)


BACKBONE: cahya/roberta-base-indonesian-1.5G
  max_len=160  batch=8  lr=1e-05  epochs=5


tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/556 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  [fold 3] epoch 2/5  train_loss=1.0854  val_bal_acc=0.6537


/tmp/ipykernel_2500/1216671756.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
/tmp/ipykernel_2500/1216671756.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


  [fold 3] epoch 3/5  train_loss=0.8289  val_bal_acc=0.6503


/tmp/ipykernel_2500/1216671756.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


KeyboardInterrupt: 

In [ ]:
# CELL 10 — Backbone D: XLM-R base
MODEL_C = 'xlm-roberta-base'
RESULTS['xlmr-base'] = run_backbone(
    MODEL_C,
    max_len=160, batch_size=16, lr=2e-5, epochs=4,
    warmup_ratio=0.1, weight_decay=0.01, grad_accum=1,
)
RESULTS['xlmr-base']['metrics'] = evaluate(
    MODEL_C,
    RESULTS['xlmr-base']['oof_probs'],
    RESULTS['xlmr-base']['fold_scores'],
)

In [8]:
# CELL 11 — Backbone E: XLM-R LARGE
# Notes:
#   - lr=1e-5 for stability on the large model.
#   - epochs=5.
#   - On T4 you will almost certainly need batch_size=8 grad_accum=2.
#   - On A100/L4 you can try batch_size=16 grad_accum=1.
MODEL_D = 'xlm-roberta-large'
RESULTS['xlmr-large'] = run_backbone(
    MODEL_D,
    max_len=160, batch_size=8, lr=1e-5, epochs=5,
    warmup_ratio=0.1, weight_decay=0.01, grad_accum=2,
)
RESULTS['xlmr-large']['metrics'] = evaluate(
    MODEL_D,
    RESULTS['xlmr-large']['oof_probs'],
    RESULTS['xlmr-large']['fold_scores'],
)


BACKBONE: xlm-roberta-large
  max_len=160  batch=8  lr=1e-05  epochs=5


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_2500/1216671756.py:58: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_2500/1216671756.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
/tmp/ipykernel_2500/1216671756.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):

  [fold 1] epoch 1/5  train_loss=2.0491  val_bal_acc=0.2882


/tmp/ipykernel_2500/1216671756.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


KeyboardInterrupt: 

In [9]:
# CELL 12 — Stacking head (transformer [CLS] embeddings → XGBoost)

STACK_BACKBONE = 'indolem/indobertweet-base-uncased'
STACK_KEY = 'stack-indobertweet-xgb'

!pip install -q xgboost
from xgboost import XGBClassifier

def extract_cls(model, tokenizer, texts, max_len, batch_size):
    ds = TweetDataset(texts, tokenizer, max_len)
    ld = DataLoader(ds, batch_size=batch_size*2, shuffle=False,
                    num_workers=2, pin_memory=True)
    model.eval()
    chunks = []
    with torch.no_grad():
        for batch in ld:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            ids = batch['input_ids']; am = batch['attention_mask']
            with torch.cuda.amp.autocast(dtype=torch.float16):
                out = model.base_model(input_ids=ids, attention_mask=am)
            cls = out.last_hidden_state[:, 0, :].float().cpu().numpy()
            chunks.append(cls)
    return np.concatenate(chunks, axis=0)


def run_stacking(model_name, *, max_len=128, batch_size=16, lr=2e-5,
                 epochs=4, warmup_ratio=0.1, weight_decay=0.05, grad_accum=1):
    print(f'\n{"="*70}\nSTACKING: {model_name} → XGBoost\n{"="*70}')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros((len(labeled), N_CLASSES), dtype=np.float32)
    test_p = np.zeros((len(predict_df), N_CLASSES), dtype=np.float32)
    fold_scores = []
    test_texts = predict_df['text'].values

    for fold, (tr_idx, va_idx) in enumerate(
            skf.split(labeled['text'], labeled['y']), 1):
        print(f'\n--- Stacking fold {fold}/{N_FOLDS} ---')
        # 1. Fine-tune backbone for this fold
        model, _ = train_one_fold(
            model_name, tokenizer, tr_idx, va_idx, fold,
            max_len=max_len, batch_size=batch_size, lr=lr, epochs=epochs,
            warmup_ratio=warmup_ratio, weight_decay=weight_decay,
            grad_accum=grad_accum,
        )
        # 2. Extract [CLS] embeddings for the fold's train / val / full test
        Xtr = extract_cls(model, tokenizer, labeled.loc[tr_idx, 'text'].values,
                          max_len, batch_size)
        Xva = extract_cls(model, tokenizer, labeled.loc[va_idx, 'text'].values,
                          max_len, batch_size)
        Xte = extract_cls(model, tokenizer, test_texts, max_len, batch_size)
        # Free GPU before fitting XGBoost
        del model; gc.collect(); torch.cuda.empty_cache()

        # 3. XGBoost classifier on the embeddings
        xgb = XGBClassifier(
            n_estimators=400, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            objective='multi:softprob', num_class=N_CLASSES,
            tree_method='hist', random_state=SEED,
            eval_metric='mlogloss',
        )
        # Sample weights to compensate for class imbalance — gives XGBoost
        # the same advantage class_weight gave the transformer.
        sw = np.array([weights_np[y] for y in labeled.loc[tr_idx, 'y'].values])
        xgb.fit(Xtr, labeled.loc[tr_idx, 'y'].values, sample_weight=sw)

        oof[va_idx] = xgb.predict_proba(Xva)
        test_p += xgb.predict_proba(Xte) / N_FOLDS
        fold_acc = balanced_accuracy_score(
            labeled.loc[va_idx, 'y'].values, oof[va_idx].argmax(1))
        fold_scores.append(fold_acc)
        print(f'  [stack fold {fold}] XGBoost val bal_acc = {fold_acc:.4f}')

    return {
        'name': f'{model_name} + XGBoost',
        'oof_probs': oof,
        'test_probs': test_p,
        'fold_scores': fold_scores,
        'config': dict(max_len=max_len, batch_size=batch_size, lr=lr,
                       epochs=epochs, warmup_ratio=warmup_ratio,
                       weight_decay=weight_decay, grad_accum=grad_accum,
                       head='xgboost'),
    }


RESULTS[STACK_KEY] = run_stacking(
    STACK_BACKBONE,
    max_len=128, batch_size=16, lr=2e-5, epochs=4,
    warmup_ratio=0.1, weight_decay=0.05, grad_accum=1,
)
RESULTS[STACK_KEY]['metrics'] = evaluate(
    RESULTS[STACK_KEY]['name'],
    RESULTS[STACK_KEY]['oof_probs'],
    RESULTS[STACK_KEY]['fold_scores'],
)


STACKING: indolem/indobertweet-base-uncased → XGBoost


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(



--- Stacking fold 1/5 ---


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobertweet-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_2500/1216671756.py:58: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_2500/1216671756.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
/tmp/ipykernel_2500/1216671756.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


  [fold 1] epoch 1/4  train_loss=1.6292  val_bal_acc=0.6442


...

  [fold 5] epoch 3/4  train_loss=0.6793  val_bal_acc=0.6546


/tmp/ipykernel_2500/1216671756.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
/tmp/ipykernel_2500/1216671756.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


  [fold 5] epoch 4/4  train_loss=0.5365  val_bal_acc=0.6478
  [fold 5] best epoch = 3, best bal_acc = 0.6546


/tmp/ipykernel_2500/1252053731.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
/tmp/ipykernel_2500/1252053731.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):


  [stack fold 5] XGBoost val bal_acc = 0.6254

EVALUATION: indolem/indobertweet-base-uncased + XGBoost
Per-fold balanced_acc : [np.float64(0.6709), np.float64(0.6411), np.float64(0.6749), np.float64(0.6766), np.float64(0.6254)]
Mean fold balanced_acc: 0.6578  (std 0.0207)

--- OOF (out-of-fold) metrics ---
  Balanced accuracy : 0.6577  <-- COMPETITION METRIC
  Plain accuracy    : 0.6655
  Macro F1          : 0.6613
  Weighted F1       : 0.6652
  Macro precision   : 0.6652
  Macro recall      : 0.6577
  Cohen kappa       : 0.6044
  Matthews corrcoef : 0.6044
  MAE (on label ids): 0.9660   (informational)
  MSE (on label ids): 3.8206   (informational)

--- Per-class report ---
                  precision    recall  f1-score   support

        Anggaran     0.7617    0.7827    0.7720       727
      Distribusi     0.6722    0.6536    0.6628       433
         Ekonomi     0.7687    0.7103    0.7384       145
 Kualitas Pangan     0.7184    0.7287    0.7235      1246
         Lainnya     0.58

---
## Comparison & selection

Cell 11: side-by-side summary table.

Cell 12: tries every 2-way, 3-way, and full-set ensemble and reports them alongside the singles.

Cell 13: **YOU pick** which model or combination to submit.

In [10]:
# CELL 13 — Summary table across all trained backbones
if not RESULTS:
    print('No models trained yet — run cells 7–10 first.')
else:
    rows = []
    for key, info in RESULTS.items():
        m = info['metrics']
        rows.append({
            'key': key,
            'model': info['name'],
            'bal_acc (OOF)': round(m['balanced_accuracy'], 4),
            'accuracy': round(m['accuracy'], 4),
            'macro_f1': round(m['macro_f1'], 4),
            'weighted_f1': round(m['weighted_f1'], 4),
            'macro_recall': round(m['macro_recall'], 4),
            'kappa': round(m['cohen_kappa'], 4),
            'mcc': round(m['mcc'], 4),
            'mean fold bal_acc': round(float(np.mean(info['fold_scores'])), 4),
            'std fold bal_acc':  round(float(np.std(info['fold_scores'])), 4),
        })
    summary = pd.DataFrame(rows).sort_values('bal_acc (OOF)', ascending=False)
    print('Per-model comparison (sorted by OOF balanced accuracy):\n')
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        print(summary.to_string(index=False))
    print(f'\nBest single model: {summary.iloc[0]["key"]} '
          f'({summary.iloc[0]["bal_acc (OOF)"]} OOF balanced accuracy)')

Per-model comparison (sorted by OOF balanced accuracy):

                    key                                       model  bal_acc (OOF)  accuracy  macro_f1  weighted_f1  macro_recall  kappa    mcc  mean fold bal_acc  std fold bal_acc
           indobertweet           indolem/indobertweet-base-uncased         0.6893    0.6749    0.6759       0.6752        0.6893 0.6180 0.6185             0.6893            0.0136
indonesian-roberta-base      flax-community/indonesian-roberta-base         0.6593    0.6461    0.6459       0.6468        0.6593 0.5842 0.5847             0.6593            0.0169
 stack-indobertweet-xgb indolem/indobertweet-base-uncased + XGBoost         0.6577    0.6655    0.6613       0.6652        0.6577 0.6044 0.6044             0.6578            0.0207

Best single model: indobertweet (0.6893 OOF balanced accuracy)


In [11]:
# CELL 14 — Try every ensemble (equal-weight soft vote) and report
from itertools import combinations

if not RESULTS:
    print('No models trained yet — run cells 7–10 first.')
else:
    keys = list(RESULTS.keys())
    y_true = labeled['y'].values

    ens_rows = []
    for k in keys:
        oof = RESULTS[k]['oof_probs']
        ens_rows.append({
            'combo': k,
            'n_models': 1,
            'bal_acc (OOF)': round(balanced_accuracy_score(y_true, oof.argmax(1)), 4),
            'macro_f1': round(f1_score(y_true, oof.argmax(1), average='macro'), 4),
        })
    for r in range(2, len(keys) + 1):
        for combo in combinations(keys, r):
            stacked = np.mean([RESULTS[k]['oof_probs'] for k in combo], axis=0)
            pred = stacked.argmax(1)
            ens_rows.append({
                'combo': ' + '.join(combo),
                'n_models': r,
                'bal_acc (OOF)': round(balanced_accuracy_score(y_true, pred), 4),
                'macro_f1': round(f1_score(y_true, pred, average='macro'), 4),
            })
    ens_df = pd.DataFrame(ens_rows).sort_values('bal_acc (OOF)', ascending=False)
    print('All single models + every soft-vote ensemble combination '
          '(sorted by OOF balanced accuracy):\n')
    with pd.option_context('display.max_columns', None, 'display.width', 200,
                           'display.max_rows', None):
        print(ens_df.to_string(index=False))
    print(f'\nBest combination: {ens_df.iloc[0]["combo"]} '
          f'({ens_df.iloc[0]["bal_acc (OOF)"]})')
    print('\nNote: ensemble "OOF balanced accuracy" here is computed by averaging '
          'each model\'s OOF softmax probabilities and taking the argmax — a '
          'reliable estimate of what equal-weight soft-vote will score on the '
          'real test set.')

All single models + every soft-vote ensemble combination (sorted by OOF balanced accuracy):

                                                          combo  n_models  bal_acc (OOF)  macro_f1
                         indobertweet + indonesian-roberta-base         2         0.6914    0.6793
                                                   indobertweet         1         0.6893    0.6759
indobertweet + indonesian-roberta-base + stack-indobertweet-xgb         3         0.6825    0.6775
                          indobertweet + stack-indobertweet-xgb         2         0.6754    0.6734
               indonesian-roberta-base + stack-indobertweet-xgb         2         0.6747    0.6731
                                        indonesian-roberta-base         1         0.6593    0.6459
                                         stack-indobertweet-xgb         1         0.6577    0.6613

Best combination: indobertweet + indonesian-roberta-base (0.6914)

Note: ensemble "OOF balanced accuracy" here is 

In [12]:
# CELL 15 — FINAL: choose which model(s) to submit
#

SELECTION = ['indobertweet', 'indoroberta']
WEIGHTS   = None
OUT_PATH  = '/content/drive/MyDrive/satria/Adinar Tri Panuntun_FOTOIN QR ABSENNYA WOI!!.xlsx'

# -----------------------------------------------------------------------
if not RESULTS:
    raise RuntimeError('No models trained yet — run cells 7–12 first.')

available = [k for k in SELECTION if k in RESULTS]
missing = [k for k in SELECTION if k not in RESULTS]
if missing:
    print(f'WARNING: these keys are not in RESULTS and will be skipped: {missing}')
if not available:
    raise RuntimeError(f'None of SELECTION={SELECTION} are in RESULTS={list(RESULTS.keys())}')

if WEIGHTS is None:
    w = np.ones(len(available)) / len(available)
else:
    if len(WEIGHTS) != len(SELECTION):
        raise ValueError('WEIGHTS length must equal SELECTION length')
    # Only keep weights matching available models, then renormalize
    w = np.array([WEIGHTS[SELECTION.index(k)] for k in available], dtype=float)
    w = w / w.sum()

print(f'Using {len(available)} model(s):')
for k, wi in zip(available, w):
    print(f'  {k:<25s}  weight = {wi:.4f}')

# Weighted soft vote on test set
test_probs = np.zeros((len(predict_df), N_CLASSES), dtype=np.float32)
oof_probs  = np.zeros((len(labeled),    N_CLASSES), dtype=np.float32)
for k, wi in zip(available, w):
    test_probs += wi * RESULTS[k]['test_probs']
    oof_probs  += wi * RESULTS[k]['oof_probs']

# Sanity check: OOF balanced accuracy of the chosen combination
y_true = labeled['y'].values
oof_pred = oof_probs.argmax(1)
print(f'\nOOF balanced accuracy of selected combination: '
      f'{balanced_accuracy_score(y_true, oof_pred):.4f}')
print(f'OOF accuracy        : {accuracy_score(y_true, oof_pred):.4f}')
print(f'OOF macro F1        : {f1_score(y_true, oof_pred, average="macro"):.4f}')
print(f'\nPer-class report on OOF:')
print(classification_report(y_true, oof_pred, target_names=LABELS, digits=4, zero_division=0))

# Make sure the output directory exists on Drive
import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

# Write submission
test_labels = [id2label[i] for i in test_probs.argmax(1)]
out = template.copy()
id_to_pred = dict(zip(predict_df['id'], test_labels))
out['label'] = out['id'].map(id_to_pred)
assert out['label'].isnull().sum() == 0, 'Some test IDs missing predictions'
out.to_excel(OUT_PATH, index=False)

print(f'\nWrote: {OUT_PATH}')
print(f'Prediction distribution:')
print(out['label'].value_counts())
print(f'\nHead:'); print(out.head())

# (Auto-download dimatikan karena file sudah otomatis tersimpan di Drive)

Using 1 model(s):
  indobertweet               weight = 1.0000

OOF balanced accuracy of selected combination: 0.6893
OOF accuracy        : 0.6749
OOF macro F1        : 0.6759

Per-class report on OOF:
                  precision    recall  f1-score   support

        Anggaran     0.7585    0.7992    0.7783       727
      Distribusi     0.6555    0.7252    0.6886       433
         Ekonomi     0.7202    0.8345    0.7732       145
 Kualitas Pangan     0.7706    0.6846    0.7250      1246
         Lainnya     0.6057    0.6019    0.6038       638
         Politik     0.6393    0.5997    0.6189       792
Sasaran Penerima     0.6138    0.6489    0.6309       507
     Tata Kelola     0.5601    0.6204    0.5887       511

        accuracy                         0.6749      4999
       macro avg     0.6655    0.6893    0.6759      4999
    weighted avg     0.6781    0.6749    0.6752      4999


Wrote: /content/drive/MyDrive/satria/nama_tim.xlsx
Prediction distribution:
label
Kualitas Pangan 